# 03 — Descomposicion estacional (HICP Eurozona)

Cuantificar la componente estacional del HICP para decidir si D=1 en SARIMA es necesario.

**Metodos:** Descomposicion clasica (aditiva + multiplicativa) | STL (LOESS) robusto a outliers | Fuerza estacional Fs

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose, STL

NOTEBOOK_DIR = Path(r"c:/Users/usuario/OneDrive/Documentos/GitHub/tfg-ipc-mcp/tfg-forecasting/02_eda")
ROOT = NOTEBOOK_DIR.parent
MONOREPO = ROOT.parent
sys.path.insert(0, str(MONOREPO))
from shared.constants import DATE_TRAIN_END
plt.rcParams.update({"figure.figsize": (14, 8), "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
df = pd.read_parquet(ROOT / "data" / "processed" / "hicp_europe_index.parquet")
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
train = df.loc[:DATE_TRAIN_END]
y = train["hicp_index"]
y.index.freq = "MS"
print(f"Train: {len(y)} obs, {y.index.min().date()} -> {y.index.max().date()}")

## 1. Descomposicion clasica — aditiva

In [ ]:
decomp_add = seasonal_decompose(y, model="additive", period=12)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp_add.observed.plot(ax=axes[0], title="Observado")
decomp_add.trend.plot(ax=axes[1], title="Tendencia")
decomp_add.seasonal.plot(ax=axes[2], title="Estacionalidad")
decomp_add.resid.plot(ax=axes[3], title="Residuo")
fig.suptitle("Descomposicion clasica ADITIVA — HICP Eurozona (periodo=12)", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## 2. Descomposicion clasica — multiplicativa

In [ ]:
decomp_mul = seasonal_decompose(y, model="multiplicative", period=12)
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp_mul.observed.plot(ax=axes[0], title="Observado")
decomp_mul.trend.plot(ax=axes[1], title="Tendencia")
decomp_mul.seasonal.plot(ax=axes[2], title="Estacionalidad (factor)")
decomp_mul.resid.plot(ax=axes[3], title="Residuo")
fig.suptitle("Descomposicion clasica MULTIPLICATIVA — HICP Eurozona (periodo=12)", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## 3. STL (LOESS) — robusto a outliers

In [ ]:
stl = STL(y, period=12, robust=True)
stl_result = stl.fit()
fig = stl_result.plot()
fig.set_size_inches(14, 10)
fig.suptitle("Descomposicion STL (LOESS, robusto) — HICP Eurozona", fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

## 4. Fuerza de la estacionalidad (Fs)

Metrica de Wang, Smith & Hyndman (2006):

$F_s = 1 - \frac{\text{Var}(R_t)}{\text{Var}(S_t + R_t)}$

- $F_s > 0.64$ indica estacionalidad significativa

In [ ]:
seasonal = stl_result.seasonal
resid = stl_result.resid
Fs = 1 - np.var(resid) / np.var(seasonal + resid)
print(f"Fuerza estacional (Fs): {Fs:.4f}")
print(f"Estacionalidad {'SIGNIFICATIVA' if Fs > 0.64 else 'NO significativa'} (umbral 0.64)")

## 5. Patron estacional mensual

In [ ]:
seasonal_pattern = stl_result.seasonal.groupby(stl_result.seasonal.index.month).mean()
months = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
colors = ["#d62728" if v > 0 else "#1565c0" for v in seasonal_pattern.values]
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(1, 13), seasonal_pattern.values, color=colors)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)
ax.set_title("Componente estacional media por mes (STL) — HICP Eurozona")
ax.set_ylabel("Efecto estacional (puntos de indice)")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()

print("Patron estacional mensual:")
for m, name in enumerate(months, 1):
    print(f"  {name}: {seasonal_pattern[m]:+.4f}")

## 6. Estabilidad temporal de la estacionalidad

In [ ]:
s = stl_result.seasonal.copy()
df_s = pd.DataFrame({"year": s.index.year, "month": s.index.month, "seasonal": s.values})
pivot = df_s.pivot(index="month", columns="year", values="seasonal")
months = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
fig, ax = plt.subplots(figsize=(14, 6))
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col], alpha=0.25, linewidth=0.8, color="grey")
ax.plot(pivot.index, pivot.mean(axis=1), linewidth=2.5, color="#d62728", label="Media")
ax.set_xticks(range(1, 13)); ax.set_xticklabels(months)
ax.set_title("Estabilidad del patron estacional — HICP Eurozona")
ax.set_ylabel("Componente estacional"); ax.legend()
plt.tight_layout(); plt.show()

## 7. Conclusion

**Resultados:**
- STL: tendencia creciente con aceleracion post-2021 (crisis inflacionaria)
- Fs: cuantifica si la estacionalidad justifica D=1 en SARIMA
- Si Fs > 0.64, incluir diferenciacion estacional

*Completar con valores reales de Fs y patron mensual.*